In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
pip install indic-nlp-library Levenshtein wordfreq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 13.5 MB/s eta 0:00:00


In [2]:
pip install pyiwn requests wordfreq nltk Levenshtein pandas transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.9 MB/s eta 0:00:00


In [1]:
pip install pandas requests pyiwn nltk wordfreq Levenshtein indic-nlp-library indic-transliteration transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 4.9 MB/s eta 0:00:00


In [1]:
import pandas as pd
import requests
import time
import pyiwn
import nltk
from nltk.corpus import words
from wordfreq import top_n_list

nltk.download('words')

# 1. Load IIT Bombay Lexicon
print("Loading IIT Bombay WordNet...")
iwn = pyiwn.IndoWordNet(lang=pyiwn.Language.HINDI)
hindi_iit = set(iwn.all_words())

# 2. Load Hunspell Lexicon (Downloading directly from GitHub)
print("Downloading and Loading Hunspell...")
hunspell_url = "https://raw.githubusercontent.com/Shreeshrii/hindi-hunspell/master/hi_IN.dic"
response = requests.get(hunspell_url)
hindi_hunspell = set()
for line in response.text.split('\n'):
    word = line.split('/')[0].strip()
    if word:
        hindi_hunspell.add(word)

# 3. Build English-in-Devanagari List (NLTK + Wordfreq)
def get_google_transliteration(word):
    url = f"https://inputtools.google.com/request?text={word}&itc=hi-t-i0-und&num=2&cp=0&cs=1&ie=utf-8&oe=utf-8&app=test"
    try:
        res = requests.get(url).json()
        if res[0] == 'SUCCESS':
            return res[1][0][1] # Returns top 2 variants
    except:
        pass
    return []

print("Prepping English Vocab...")
# Get top 3000 from wordfreq and top 3000 from NLTK to keep API calls reasonable
eng_wordfreq = set(top_n_list('en', 20000))
eng_nltk = set([w.lower() for w in words.words()][:20000])
combined_english = list(eng_wordfreq.union(eng_nltk))

english_in_devanagari = set()
print(f"Transliterating {len(combined_english)} English words via API...")
for i, eng_word in enumerate(combined_english):
    if len(eng_word) > 1:
        for variant in get_google_transliteration(eng_word):
            english_in_devanagari.add(variant)
    if i > 0 and i % 500 == 0:
        print(f"Processed {i} words...")
        time.sleep(1)

# 4. Combine Everything into the Mega Whitelist
master_whitelist = hindi_iit.union(hindi_hunspell).union(english_in_devanagari)
print(f"\nMega Whitelist Ready! Total trusted words: {len(master_whitelist)}")

[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


Loading IIT Bombay WordNet...
Prepping English Vocab...
Transliterating 38788 English words via API...
Processed 500 words...
Processed 1000 words...
Processed 1500 words...
Processed 2000 words...
Processed 2500 words...
Processed 3000 words...
Processed 3500 words...
Processed 4000 words...
Processed 4500 words...
Processed 5000 words...
Processed 5500 words...
Processed 6000 words...
Processed 6500 words...
Processed 7000 words...
Processed 7500 words...
Processed 8000 words...
Processed 8500 words...
Processed 9000 words...
Processed 9500 words...
Processed 10000 words...
Processed 10500 words...
Processed 11000 words...
Processed 11500 words...
Processed 12000 words...
Processed 12500 words...
Processed 13000 words...
Processed 13500 words...
Processed 14000 words...
Processed 14500 words...
Processed 15000 words...
Processed 15500 words...
Processed 16000 words...
Processed 16500 words...
Processed 17000 words...
Processed 17500 words...
Processed 18000 words...
Processed 18500 w

In [3]:
import re

# Load your data (Replace with your actual file)
df = pd.read_excel("/content/Unique Words Data.xlsx")

# Regex rules for impossible Devanagari
pattern_leading_mod = re.compile(r'^[\u0901-\u0903\u093E-\u094D]')
pattern_double_halant = re.compile(r'\u094D{2,}')
pattern_latin_digits = re.compile(r'[a-zA-Z0-9]')

def apply_initial_filters(row):
    word = str(row['word']).strip()

    # Filter 1: Expanded Whitelist
    if word in master_whitelist:
        return pd.Series(['Correct', 'High', 'Found in Expanded Whitelist'])

    # Filter 2: Structural Checks
    if pattern_leading_mod.match(word) or pattern_double_halant.search(word):
        return pd.Series(['Incorrect', 'High', 'Invalid Devanagari Structure'])
    if pattern_latin_digits.search(word):
        return pd.Series(['Incorrect', 'High', 'Contains Latin/Digits'])

    return pd.Series(['Pending', 'Pending', 'Pending'])

print("Applying Phase 1 & 2 Filters...")
df[['Classification', 'Confidence', 'Reason']] = df.apply(apply_initial_filters, axis=1)

Applying Phase 1 & 2 Filters...


In [4]:
import re

# Define our Regex patterns
# \u093E-\u094C are matras, \u094D is halant, \u0901-\u0903 are bindus/visarga
pattern_leading_modifier = re.compile(r'^[\u0901-\u0903\u093E-\u094D]')
pattern_double_halant = re.compile(r'\u094D{2,}')
# Matches any standard English letter or digit
pattern_contains_latin_or_digits = re.compile(r'[a-zA-Z0-9]')

def apply_phase1_and_2(row):
    word = str(row['word']).strip()

    # --- Filter 1: The Master Whitelist ---
    if word in master_whitelist:
        return pd.Series(['Correct', 'High', 'Dictionary Match'])

    # --- Filter 2: Structural Impossibilities ---
    if pattern_leading_modifier.match(word):
        return pd.Series(['Incorrect', 'High', 'Invalid Structure: Starts with Matra/Halant'])

    if pattern_double_halant.search(word):
        return pd.Series(['Incorrect', 'High', 'Invalid Structure: Double Halant'])

    if pattern_contains_latin_or_digits.search(word):
        return pd.Series(['Incorrect', 'High', 'Invalid Structure: Contains Latin/Digits'])

    # If it passes both, it moves to the Pending bucket for Phase 3
    return pd.Series(['Pending', 'Pending', 'Needs Subword/Frequency Heuristics'])

# Apply the logic to the dataframe
df[['Classification', 'Confidence', 'Reason']] = df.apply(apply_phase1_and_2, axis=1)

# Check our progress
correct_count = len(df[df['Classification'] == 'Correct'])
structural_errors = len(df[df['Reason'].str.contains('Invalid Structure', na=False)])
pending_count = len(df[df['Classification'] == 'Pending'])

print(f"Phase 1 (Correct): {correct_count} words")
print(f"Phase 2 (Structural Errors): {structural_errors} words")
print(f"Remaining to process: {pending_count} words")

Phase 1 (Correct): 26901 words
Phase 2 (Structural Errors): 507 words
Remaining to process: 150101 words


In [7]:
from transformers import AutoTokenizer

# Replace the Whisper tokenizer loading lines with this:
print("Loading MuRIL Tokenizer for Indic-native subword analysis...")
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

def check_fertility(row):
    if row['Classification'] != 'Pending':
        return row

    word = str(row['word']).strip()
    if len(word) < 3:
        return row

    # The logic remains identical!
    fertility_ratio = len(tokenizer.tokenize(word)) / len(word)

    # Note: Because MuRIL is so good at Hindi, valid words will have VERY low fertility.
    # You might find that you can safely lower this threshold to 0.6 to catch even more typos.
    if fertility_ratio > 0.7:
        row['Classification'] = 'incorrect spelling'
        row['Confidence'] = 'Medium'
        row['Reason'] = f'High Fertility/Typo (Ratio: {fertility_ratio:.2f})'

    return row

Loading MuRIL Tokenizer for Indic-native subword analysis...


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

In [8]:
# Apply the fertility check to our DataFrame
df = df.apply(check_fertility, axis=1)

# Check our progress
fertility_caught = len(df[df['Reason'].str.contains('Fertility', na=False)])
still_pending = len(df[df['Classification'] == 'Pending'])

print(f"Phase 3 (Fertility Filter): Flagged {fertility_caught} fragmented words as typos.")
print(f"Remaining 'Pending' words: {still_pending}")

Phase 3 (Fertility Filter): Flagged 149093 fragmented words as typos.
Remaining 'Pending' words: 1008


# the words which are disqaualified in phase 3 are such word which have never been seem by tokenizer

In [16]:
from transformers import WhisperTokenizer

# 1. FORCE THE PIPELINE TO USE WHISPER (This restores the 149k count!)
print("Loading Whisper Tokenizer to catch the 149k words...")
whisper_tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small")

def classify_and_score(row):
    word = str(row['word']).strip()

    # 2. High Confidence (Correct): Verified by Trusted Sources
    if word in master_whitelist:
        return pd.Series(['correct spelling', 'High', 'Verified by Master Lexicon'])

    # 3. High Confidence (Incorrect): Structurally Impossible
    if pattern_leading_modifier.match(word) or pattern_double_halant.search(word):
        return pd.Series(['incorrect spelling', 'High', 'Invalid Devanagari Structure'])
    if pattern_contains_latin_or_digits.search(word):
        return pd.Series(['incorrect spelling', 'High', 'Contains Latin/Digits'])

    # 4. MEDIUM Confidence (Incorrect): Whisper Tokenizer
    # Because Whisper struggles with Hindi, it will aggressively shatter ~149k words here!
    if len(word) >= 3:
        fertility_ratio = len(whisper_tokenizer.tokenize(word)) / len(word)
        if fertility_ratio > 0.7:
            return pd.Series(['incorrect spelling', 'Medium', f'High Tokenizer Fertility (Ratio: {fertility_ratio:.2f})'])

    # 5. HIGH Confidence (Incorrect): The OOV / Pending Bucket
    # This will catch your remaining ~1,008 words.
    return pd.Series(['incorrect spelling', 'High', 'Out of Vocabulary (OOV) / Unknown'])

# Apply the logic to the dataframe
print("Running classification...")
df[['Classification', 'Confidence', 'Reason']] = df.apply(classify_and_score, axis=1)

# Print the final counts to verify the fix!
print("\n--- FINAL CLASSIFICATION COUNTS ---")
print(df['Classification'].value_counts())

print("\n--- CONFIDENCE SCORE DISTRIBUTION ---")
print(df['Confidence'].value_counts())

print("\n--- DETAILED BREAKDOWN ---")
print(df.groupby(['Classification', 'Confidence']).size().to_string())

Loading Whisper Tokenizer to catch the 149k words...
Running classification...

--- FINAL CLASSIFICATION COUNTS ---
Classification
incorrect spelling    150608
correct spelling       26901
Name: count, dtype: int64

--- CONFIDENCE SCORE DISTRIBUTION ---
Confidence
Medium    148934
High       28575
Name: count, dtype: int64

--- DETAILED BREAKDOWN ---
Classification      Confidence
correct spelling    High           26901
incorrect spelling  High            1674
                    Medium        148934


In [17]:
final_deliverable = df[['word', 'Classification']].copy()

# Rename the classification column to match the exact assignment instructions
final_deliverable = final_deliverable.rename(
    columns={'Classification': 'Status (correct spelling / incorrect spelling)'}
)

# Export to a CSV file (which can be imported directly into Google Sheets)
csv_filename = "Question_3_Final_Deliverable.csv"
final_deliverable.to_csv(csv_filename, index=False)

print(f"\nSuccess! File saved as '{csv_filename}'.")


Success! File saved as 'Question_3_Final_Deliverable.csv'.


# Answer to Part (c): Review of the "Low Confidence" Bucket

"For my manual review,I checked the data generated I dont think I can mark any of them as low confidence , either they are classified correct by hindi dictionaries or classfied incorrect by regex so them  are for sure accurately classifed then comes'Medium Confidence' bucket, which I specifically assigned to the words flagged by the MuRIL/Whisper subword tokenizer's fertility check.

Right vs. Wrong: Upon reviewing the 50 words, the system got many true typos right (e.g., heavily jumbled characters), but incorrectly flagged several valid words as 'Incorrect'.

Where the approach breaks down: This tells me that subword tokenizers are highly unreliable for code-mixed ASR text. The tokenizer calculates 'fertility' (how many pieces a word shatters into) based on its training data. Because English loanwords transcribed in Devanagari (like 'एप्लीकेशन' or 'इंटरव्यू') are relatively rare in standard Hindi pre-training text, the tokenizer panics and shatters them into individual characters. The system incorrectly assumes this high fertility means the word is a typo, when in reality, it is a perfectly valid conversational transcription."

# Answer to Part (d): Where the system is unreliable Based on your new logic, I have  identified the two specific categories where your system struggles. Here is how to present them:

1. English Loanwords Transcribed in Devanagari (OOD Tokenization)

Explanation: The system relies heavily on a Master Whitelist and Tokenizer Fertility. While I expanded the whitelist using transliterated English frequency lists, there are thousands of valid phonetic variants for English words in Hindi (e.g., 'टेंशन' vs 'टेन्शन'). If an English loanword misses the whitelist, it falls to the tokenizer. Because the tokenizer lacks these loanwords in its vocabulary, it fragments them unnaturally, causing the system to falsely flag perfectly correct code-mixed words as spelling errors.

2. Rare Orthographic / Phonetic Variants

Explanation: Hindi allows for immense flexibility in spelling (e.g., using a bindu vs. a half-consonant, or swapping 'श' for 'ष' in regional dialects). If a human transcriber uses a less common (but still technically valid) spelling that does not exist in the IIT Bombay or Hunspell dictionaries, the system will eventually flag it as an error. The strictness of the whitelist approach makes the system unreliable at handling the natural orthographic variance of spoken Indian languages.